# PIPK-Net Tutorial — Predicting Pharmacokinetic Parameters

**PIPK-Net** is a physiology-informed graph neural network that predicts seven systemic
pharmacokinetic (PK) parameters for orally administered small molecules directly from
their structure:

| Parameter | Unit | Meaning |
|---|---|---|
| `t_half (h)` | h | Elimination half-life |
| `Vd (L)` | L | Volume of distribution |
| `CL (L/h)` | L/h | Systemic clearance |
| `F (%)` | % | Oral bioavailability |
| `PPB (%)` | % | Plasma protein binding |
| `Cmax (uM)` | µM | Peak plasma concentration |
| `Tmax (h)` | h | Time to peak concentration |

It ships three trained variants; this tutorial uses the recommended one:

- **`A_baseline`** — molecular graph only.
- **`B_ion`** — graph + a learnable **ionisation-state embedding**.
- **`C_physio`** *(recommended)* — `B_ion` plus a **mass-balance loss** enforcing
  *t½ = ln(2) · Vd / CL*.

Each variant is a **5-fold ensemble**: predictions are reported as the ensemble
**mean ± standard deviation** (a simple model-uncertainty estimate).

## 1. Setup

Load the ensemble once with `PIPKNetPredictor`, then reuse it for every prediction.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from pipknet import PIPKNetPredictor

# Locate the repository root (works whether the notebook is opened from examples/ or the root)
REPO = Path.cwd()
while not (REPO / "checkpoints").exists() and REPO != REPO.parent:
    REPO = REPO.parent

predictor = PIPKNetPredictor(REPO / "checkpoints" / "C_physio")
print(f"Loaded variant '{predictor.variant}' with {len(predictor.models)} folds on {predictor.device}")

C:\Users\ACER\miniconda3\envs\pk_project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded variant 'C_physio' with 5 folds on cuda


## 2. Predict a single molecule

Pass a SMILES string and its ionisation state. The result is a tidy table of
mean ± standard deviation across the five folds, in physical units.

We use **selumetinib** (a MEK inhibitor; the manuscript's worked example), predicted as
`neutral` (its ionisation state at pH 7.4).

In [2]:
selumetinib = "CN1C=NC2=C1C=C(C(=C2F)NC3=C(C=C(C=C3)Br)Cl)C(=O)NOCCO"
predictor.predict(selumetinib, ion_type="neutral").round(2)

,Mean,Std
Parameter,,
t_half (h),11.19,2.16
Vd (L),172.05,38.74
CL (L/h),11.31,2.02
F (%),78.94,2.93
PPB (%),92.55,3.71
Cmax (uM),1.23,0.28
Tmax (h),2.13,0.19


Predicting `Vd`, `CL` and `t½` is PIPK-Net's main strength; the absorption-related
endpoints (`Cmax`, `Tmax`) are intrinsically harder and carry wider model uncertainty —
note the larger relative standard deviations.

## 3. Ionisation state

PIPK-Net takes the molecule's **physiological ionisation state at pH 7.4** as an input
feature: `anionic` (acid), `cationic` (base), `neutral`, or `zwitterionic`. This is a
**property of the molecule** — set by its pKa — **not a free parameter to tune**.

To classify a new compound, use a pKa prediction tool (ChemAxon's chemicalize.com,
DrugBank, or the manuscript's reference table) and select the dominant species at pH 7.4:

- `anionic` — net negative charge (carboxylic acids, e.g. ibuprofen, losartan)
- `cationic` — net positive charge (amines, e.g. venlafaxine, lidocaine)
- `neutral` — no group ionised near pH 7.4 (e.g. selumetinib, secobarbital)
- `zwitterionic` — both an acid and a base group are charged (e.g. amino-acid drugs)

Below we predict for **venlafaxine** (cationic; clinical Vd 308–525 L), a different
ionisation class from selumetinib above:

In [3]:
venlafaxine = "COc1ccc(C(CN(C)C)C2(O)CCCCC2)cc1.Cl"
predictor.predict(venlafaxine, ion_type="cationic").round(2)

,Mean,Std
Parameter,,
t_half (h),11.23,1.16
Vd (L),495.47,128.41
CL (L/h),31.88,5.61
F (%),61.58,6.54
PPB (%),82.79,1.97
Cmax (uM),0.24,0.16
Tmax (h),2.46,0.08


The predicted Vd for venlafaxine is markedly higher than for selumetinib — reflecting the
extensive tissue distribution of basic (cationic) drugs that bind to acidic tissue
components. This is the regime where PIPK-Net's ionisation embedding contributes most;
see manuscript Table 5 for the full benchmark comparison.

## 4. Batch prediction

`predict_batch` accepts a DataFrame, a list of dicts, or a path to a CSV with a `SMILES`
column (and optionally `IonType` and `Name`). It returns one row per molecule with
`<parameter>_mean` / `<parameter>_std` columns. Here we use the bundled `example_drugs.csv`,
which spans acidic, neutral and basic drugs — each predicted at its own ionisation state.

In [4]:
results = predictor.predict_batch(REPO / "example_drugs.csv")
results.to_csv("my_predictions.csv", index=False)   # save the full table

mean_cols = ["Name", "IonType", "Vd (L)_mean", "CL (L/h)_mean", "t_half (h)_mean",
             "F (%)_mean", "PPB (%)_mean"]
results[mean_cols].round(1)

,Name,IonType,Vd (L)_mean,CL (L/h)_mean,t_half (h)_mean,F (%)_mean,PPB (%)_mean
0,Selumetinib,neutral,172.000000,11.300000,11.2,78.900002,92.599998
1,Darolutamide,neutral,120.800003,10.800000,8.3,78.500000,89.699997
2,Sulfisoxazole,anionic,20.200001,2.300000,7.6,81.099998,95.099998
3,Penicillin V,anionic,26.200001,5.800000,3.2,72.500000,82.500000
4,Losartan,anionic,33.500000,4.000000,6.4,83.400002,96.199997
5,Secobarbital,anionic,37.400002,6.300000,4.2,71.000000,82.599998
6,Venlafaxine,cationic,629.700012,30.700001,14.8,62.799999,87.099998
7,Tofacitinib,cationic,386.200012,37.599998,7.4,63.200001,62.799999


## 5. Physiological consistency (mass balance)

The `C_physio` variant is trained so that its `Vd`, `CL` and `t½` predictions respect the
first-order elimination relationship **t½ = ln(2) · Vd / CL**. We can check this on the
batch: the predicted half-life should sit within a small fold-deviation of the value
implied by the predicted `Vd` and `CL`. The constraint is a **soft regulariser** applied
during training (not a hard equation), so small per-compound deviations are expected.

In [5]:
vd  = results["Vd (L)_mean"]
cl  = results["CL (L/h)_mean"]
th  = results["t_half (h)_mean"]
implied_t_half = np.log(2) * vd / cl
fold_dev = 10 ** np.abs(np.log10(th / implied_t_half))

check = pd.DataFrame({
    "Name": results["Name"],
    "t_half pred (h)": th.round(2),
    "ln2*Vd/CL (h)": implied_t_half.round(2),
    "fold deviation": fold_dev.round(2),
})
print(f"Median fold deviation from mass balance: {np.median(fold_dev):.2f}x")
check

Median fold deviation from mass balance: 1.05x


,Name,t_half pred (h),ln2*Vd/CL (h),fold deviation
0,Selumetinib,11.19,10.54,1.06
1,Darolutamide,8.32,7.76,1.07
2,Sulfisoxazole,7.59,6.21,1.22
3,Penicillin V,3.24,3.11,1.04
4,Losartan,6.44,5.82,1.11
5,Secobarbital,4.23,4.09,1.03
6,Venlafaxine,14.77,14.22,1.04
7,Tofacitinib,7.35,7.12,1.03


## 6. Comparing model variants

The physiology-informed priors help most for **highly tissue-distributed (high-Vd) basic
drugs**, which structure-only models tend to under-predict. Here we compare `A_baseline`
against `C_physio` (PIPK-Net) on several cationic drugs from the held-out test set whose
observed Vd lies in the medium-high range (300–2000 L) — the cohort where the manuscript
reports PIPK-Net's largest gains (Table 4).

In [6]:
baseline = PIPKNetPredictor(REPO / "checkpoints" / "A_baseline")

# Cationic, medium-high-Vd drugs from the held-out test set (SMILES as in e-Drug3D;
# observed Vd in L). PIPK-Net is the C_physio variant loaded in section 1.
drugs = [
    ("Venlafaxine",    "COc1ccc(C(CN(C)C)C2(O)CCCCC2)cc1.Cl",            "cationic", 487),
    ("Citalopram",     "Br.CN(C)CCCC1(c2ccc(F)cc2)OCc2cc(C#N)ccc21",     "cationic", 780),
    ("Desvenlafaxine", "CN(C)CC(c1ccc(O)cc1)C1(O)CCCCC1.O=C(O)CCC(=O)O", "cationic", 370),
    ("Pitolisant",     "Cl.Clc1ccc(CCCOCCCN2CCCCC2)cc1",                 "cationic", 700),
]
rows = []
for name, smi, ion, obs_vd in drugs:
    a = baseline.predict(smi, ion).loc["Vd (L)", "Mean"]
    c = predictor.predict(smi, ion).loc["Vd (L)", "Mean"]
    rows.append({"Drug": name, "Observed Vd (L)": obs_vd,
                 "A_baseline Vd (L)": round(a), "PIPK-Net Vd (L)": round(c)})
pd.DataFrame(rows)

,Drug,Observed Vd (L),A_baseline Vd (L),PIPK-Net Vd (L)
0,Venlafaxine,487,456,495
1,Citalopram,780,1002,766
2,Desvenlafaxine,370,123,329
3,Pitolisant,700,467,519


PIPK-Net's Vd predictions track the observed values closely across these basic drugs, while
the graph-only `A_baseline` deviates more — markedly under-predicting desvenlafaxine and
over-predicting citalopram. This is the medium-high-Vd regime where the ionisation
embedding and mass-balance constraint contribute most.

## 7. Command-line interface

The same predictions are available without writing any Python:

```bash
# single molecule
python -m pipknet.cli predict --smiles "CN1C=NC2=C1C=C(C(=C2F)NC3=C(C=C(C=C3)Br)Cl)C(=O)NOCCO" \
    --ion neutral --weights ./checkpoints/C_physio

# batch from a CSV (SMILES[, IonType, Name]) -> results.csv
python -m pipknet.cli predict --file example_drugs.csv \
    --weights ./checkpoints/C_physio --output results.csv
```

## Notes & caveats

- **Units** follow the column names: hours, litres, L/h, percentages, µM.
- **Determining IonType:** classify by the dominant species at pH 7.4 — `anionic` if it is
  net-negative (acid), `cationic` if net-positive (base), `zwitterionic` if both an acidic
  and a basic group are charged, otherwise `neutral`. Use a pKa tool (e.g. ChemAxon /
  chemicalize.com) or a database (DrugBank, ChEMBL) if unsure.
- Predictions can depend on the exact input structure (e.g. salt vs free base), since the
  training labels were keyed to specific e-Drug3D entries.
- Predictions are **label-level population estimates** (trained on FDA-label PK), not
  individual-patient values; treat the fold standard deviation as model uncertainty only.
- Accuracy is strongest for the `Vd`–`CL`–`t½` triad (the physiology-informed targets) and
  within the drug-like applicability domain; see the manuscript for the full evaluation.